## Paso 1 — Cargamos el dataset ya tipado

Mismo parquet que los notebooks anteriores. Este notebook cubre las
Fases 9 y 10 del plan (§10-11): outliers/anomalías y sesgos de
representatividad.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

INTERIM_DIR = Path("../datos/interim")
df = pd.read_parquet(INTERIM_DIR / "viajes_btura_tipado.parquet")

print(df.shape)


(41237, 17)


## Paso 2 — Outliers de flete por segmento (§10 del plan)

Aplicamos IQR 1,5x segmentado por `idMedioTransporteFo`, no de forma
global (el plan lo pide explícitamente: un valor normal para un
tractocamión puede ser extremo para una camioneta). Comparamos contra
el conteo global ya visto en el notebook `03` (51 outliers).


In [2]:
def contar_outliers_iqr_grupo(grupo):
    q1, q3 = grupo.quantile([0.25, 0.75])
    riq = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * riq, q3 + 1.5 * riq
    return ((grupo < lim_inf) | (grupo > lim_sup)).sum()

flete_valido = df[df["fleteProduccionFo"] > 0]

outliers_por_vehiculo = flete_valido.groupby("idMedioTransporteFo")["fleteProduccionFo"].apply(contar_outliers_iqr_grupo)
n_por_vehiculo = flete_valido.groupby("idMedioTransporteFo").size()

resumen = pd.DataFrame({"n": n_por_vehiculo, "outliers": outliers_por_vehiculo})
resumen["pct_outliers"] = (resumen["outliers"] / resumen["n"] * 100).round(2)
print(resumen.sort_values("pct_outliers", ascending=False))
print(f"\nTotal outliers segmentados por vehículo: {outliers_por_vehiculo.sum()}")

q1, q3 = flete_valido["fleteProduccionFo"].quantile([0.25, 0.75])
riq = q3 - q1
lim_sup_global = q3 + 1.5 * riq
outliers_global = (flete_valido["fleteProduccionFo"] > lim_sup_global).sum()
print(f"Total outliers SIN segmentar (método global, notebook 03): {outliers_global}")


                         n  outliers  pct_outliers
idMedioTransporteFo                               
TC_CAMNETA             582       107         18.38
TC-4-MANOS             225        12          5.33
TC_SENCI               940        10          1.06
FP-2S3                 290         3          1.03
TC_TURBO              1213        11          0.91
TC-3S3               22223       199          0.90
TC-3S2                3116         9          0.29
FP-2S2                 939         2          0.21
FP-3S3                2223         3          0.13
TC-2S2                7889         4          0.05
1                        2         0          0.00
TC-2S3                1306         0          0.00
FP-3S2                  29         0          0.00
TC-3S4                   6         0          0.00
TC-DOBTROQ              70         0          0.00

Total outliers segmentados por vehículo: 360
Total outliers SIN segmentar (método global, notebook 03): 51


In [3]:
camioneta = df[(df["idMedioTransporteFo"] == "TC_CAMNETA") & (df["fleteProduccionFo"] > 0)]

q1, q3 = camioneta["fleteProduccionFo"].quantile([0.25, 0.75])
riq = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * riq, q3 + 1.5 * riq
print(f"Rango normal para TC_CAMNETA: ${lim_inf:,.0f} a ${lim_sup:,.0f} (P25=${q1:,.0f}, P75=${q3:,.0f})")

outliers_camioneta = camioneta[(camioneta["fleteProduccionFo"] < lim_inf) | (camioneta["fleteProduccionFo"] > lim_sup)]
print(f"\n{outliers_camioneta.shape[0]} outliers de {camioneta.shape[0]} viajes TC_CAMNETA")
print(outliers_camioneta[["fo", "fleteProduccionFo", "distanciaTotal", "pesoNeto", "origen", "destino"]]
      .sort_values("fleteProduccionFo", ascending=False).head(15))


Rango normal para TC_CAMNETA: $850,000 a $2,050,000 (P25=$1,300,000, P75=$1,600,000)

107 outliers de 582 viajes TC_CAMNETA
              fo  fleteProduccionFo  distanciaTotal  pesoNeto  \
3784   301209171          3300000.0          1187.0     248.0   
24977  301274332          3200000.0          1116.0     701.0   
24551  301213094          3000000.0          1313.0     919.0   
25449  301281088          3000000.0          1113.0    1160.0   
31329  301271906          2450000.0          1085.0     300.0   
25429  301356296          2200000.0           497.0    2120.0   
37738  301370016          2150000.0           485.0    1000.0   
3787   301289117          2100000.0           460.0    1095.0   
24522  301383592          2100000.0           580.0     353.0   
6946   301234515           800000.0           220.0    1845.0   
5800   301400339           800000.0           145.0    1496.0   
24510  301369393           800000.0           160.0    1208.0   
12772  301384086           8000

## Síntesis del Paso 2 — Outliers de flete por segmento

Segmentar por tipo de vehículo encontró 360 outliers de flete (vs. 51
del método global del notebook `03`) — más, no menos, porque cada grupo
se compara contra su propio rango típico (mucho más angosto que el
global), y eso es justo lo que se busca: detectar qué es raro *para ese
tipo de vehículo en particular*.

**Caso investigado a fondo: `TC_CAMNETA` (18,38% de outliers, la tasa
más alta).** Se dividió en dos grupos con explicaciones opuestas:

- **Outliers altos (>$2,05M):** viajes larguísimos (1.000+ km, a Cúcuta,
  Cartagena) para una camioneta que normalmente hace recorridos cortos.
  **Clasificación: caso_legítimo** — una camioneta ocasionalmente hace
  un viaje excepcional, y cuesta más en consecuencia.
- **Outliers bajos, todos en exactamente $800.000:** investigado
  directamente en la base de datos operativa de la empresa
  (`tblSAPTarifas`, vía DBeaver). Ese valor aparece repetido en decenas
  de combinaciones ruta+vehículo+peso completamente distintas entre sí
  (desde 0 kg hasta cargas de 34 toneladas), en rutas poco frecuentes, y
  con vigencias sin patrón entre 2024 y 2026. **Conclusión: es un valor
  de relleno del sistema para combinaciones sin tarifa configurada, no
  una tarifa mínima real de negocio.** **Clasificación: error_dato /
  valor de sistema sin configurar** — mismo tipo de problema que el
  hallazgo de "Tarifa = Flete" del notebook `04`.

**Lección metodológica:** cuando un outlier estadístico resulta ser un
valor idéntico repetido en contextos muy distintos, es señal de un
defecto de sistema (valor de relleno/default), no de una coincidencia de
negocio — vale la pena verificarlo contra la fuente antes de aceptarlo
como "caso extremo legítimo".

**Limitación del método (importante):** este chequeo segmenta por una
sola variable (tipo de vehículo) — no controla distancia ni peso al
mismo tiempo. Por eso puede marcar como "raro" un valor que en realidad
está bien explicado por otra variable que no se está mirando (como pasó
con los outliers altos de `TC_CAMNETA`, explicados por distancia). Los
360 outliers segmentados son una primera pasada útil, no la palabra
final — un método multivariado (`IsolationForest`, sugerido en el plan)
podría descartar varios de ellos al considerar distancia y peso a la vez.



## Paso 3 — Revisión manual de casos puntuales

Casos señalados por el plan (§10) para revisión uno por uno, no
automática: el flete más alto, el flete negativo, y las distancias en 0
con peso/tarifa positivos.


In [4]:
print("Flete más alto:")
print(df.nlargest(5, "fleteProduccionFo")
      [["fo", "fleteProduccionFo", "distanciaTotal", "pesoNeto", "idMedioTransporteFo", "origen", "destino"]])

print("\nFlete negativo:")
print(df[df["fleteProduccionFo"] < 0]
      [["fo", "fleteProduccionFo", "distanciaTotal", "pesoNeto", "tarifaVentaAlCliente",
        "idMedioTransporteFo", "origen", "destino", "fechaCreacionFo"]])

print("\nDistancia en 0 con peso y tarifa positivos:")
print(df[(df["distanciaTotal"] == 0) & (df["pesoNeto"] > 0) & (df["tarifaVentaAlCliente"] > 0)]
      [["fo", "distanciaTotal", "pesoNeto", "tarifaVentaAlCliente", "fleteProduccionFo", "origen", "destino"]])


Flete más alto:
              fo  fleteProduccionFo  distanciaTotal  pesoNeto  \
11962  301268956         46800000.0         521.001   11752.0   
11965  301269826         46800000.0         521.001    3758.0   
11949  301266660         40500000.0         521.001    3434.0   
11960  301267947         40500000.0         521.001    6099.0   
11954  301267211         39600000.0         521.001    7236.0   

      idMedioTransporteFo        origen destino  
11962              FP-2S3  BUENAVENTURA    COTA  
11965              FP-2S3  BUENAVENTURA    COTA  
11949              TC-2S2  BUENAVENTURA    COTA  
11960              TC-2S2  BUENAVENTURA    COTA  
11954              FP-2S3  BUENAVENTURA    COTA  

Flete negativo:
              fo  fleteProduccionFo  distanciaTotal  pesoNeto  \
26152  301291984           -49978.0           498.0     467.0   

       tarifaVentaAlCliente idMedioTransporteFo        origen destino  \
26152              850000.0          TC_CAMNETA  BUENAVENTURA  GUARNE   

## Síntesis del Paso 3 — Revisión manual de casos puntuales

**Flete más alto ($39,6M-$46,8M, Buenaventura→Cota) — error_dato
CONFIRMADO.** Verificado directamente contra la tarifa oficial en la
base de datos operativa (`tblSAPTarifas`): el rango legítimo completo
para esa ruta (todos los pesos y vehículos, incluida carga plena de
30-34 ton) va de $1,1M a $7,4M. Los 5 registros sospechosos (FO
301268956, 301269826, 301266660, 301267947, 301267211) valen 5-6 veces
ese techo — imposible que sea una tarifa real. Candidato fuerte a error
de captura (cero de más o decimal corrido). Nota adicional: la distancia
de estos registros (521,001 km) rompe el patrón de "distancias limpias
de tabla" ya documentado, otra señal de anomalía. **Acción: reportar
estos FO al negocio para corrección.**

**Flete negativo (-$49.978, FO 301291984) — por_confirmar.** Caso
único, aislado, sin patrón. Explicación más plausible: nota crédito o
ajuste contable, no se puede confirmar solo con el dato disponible.

**Distancia en 0 con peso/tarifa positivos (16 registros) —
error_dato (campo faltante).** Se dividen en 2 grupos: 3 son
Buenaventura→Buenaventura (ya conocidos, segmento de descontenerización)
y 13 son rutas reales (Yumbo, Funza, Soledad, Bucaramanga, Manizales,
Itagüí...) con pesos sustanciales (4.000-34.210 kg) a las que
simplemente les falta el valor de distancia — vacío de dato, no error
de cálculo.


## Paso 4 — Concentración de clientes

¿Cuántos clientes concentran el volumen? Top 10 / total, más el índice
HHI (§11, punto 4 del plan) — si son pocos clientes grandes, cualquier
conclusión general del EDA en realidad describe a esos clientes, no al
negocio completo.


In [5]:
conteo_cliente = df["bpCliente"].value_counts()
total = conteo_cliente.sum()

top10_pct = conteo_cliente.head(10).sum() / total * 100
print(f"Top 10 clientes concentran el {top10_pct:.1f}% del volumen de viajes")

participacion = conteo_cliente / total
hhi = (participacion ** 2).sum() * 10000
print(f"Índice HHI: {hhi:.0f} (<1.500 no concentrado, 1.500-2.500 moderado, >2.500 muy concentrado)")

print("\nTop 10 clientes por volumen:")
print(pd.DataFrame({
    "viajes": conteo_cliente.head(10),
    "pct": (conteo_cliente.head(10) / total * 100).round(2)
}))


Top 10 clientes concentran el 39.7% del volumen de viajes
Índice HHI: 331 (<1.500 no concentrado, 1.500-2.500 moderado, >2.500 muy concentrado)

Top 10 clientes por volumen:
           viajes    pct
bpCliente               
580          6053  14.68
1910         2278   5.52
6            1515   3.67
761          1502   3.64
2013         1020   2.47
12503         864    2.1
1987          858   2.08
56788         853   2.07
17053         739   1.79
15961         680   1.65


## Paso 5 — Sesgo de cobertura geográfica

No calculable desde el propio dataset (por definición solo contiene
viajes de Buenaventura). Confirmado directamente con el negocio:

**Este set cubre únicamente viajes con origen o destino en
Buenaventura.** La operación de la empresa tiene muchos más orígenes y
destinos a nivel nacional, que quedan fuera de este set y se evaluarán
con otro dataset en un trabajo posterior.

**Implicación:** este EDA, y cualquier modelo predictivo que se
entrene sobre él, aplican solo al corredor de Buenaventura — no son
extrapolables a la operación nacional completa hasta que se incorpore
ese otro dataset.


## Paso 6 — Resto de sesgos (§11 del plan)

**Sesgo de selección y de supervivencia — CONFIRMADOS con evidencia
directa (verificado en `FactFWOFUFO`, tabla fuente real detrás de este
tipo de extracto).** En la misma ventana de fechas, a nivel compañía
completa: 75.969 FO en `Concluido`/`Ejecutado`, **8.034 en `Anulado`**
(7.786 ni siquiera iniciados), y **115.216 en `En tratamiento`** (más
que los concluidos). Esto confirma que `Viajes-Btura-2025-2026.xlsx`
casi con certeza solo contiene FOs `Concluido`/`Ejecutado` — deja fuera
tanto los anulados como los que seguían en curso al momento del corte.
**El dataset tiene sesgo de supervivencia: solo muestra viajes que
llegaron a un final exitoso**, no el universo completo de viajes
iniciados. (No verificado con el filtro exacto de Buenaventura —
requeriría cruce con tabla de ubicaciones — pero la evidencia a nivel
compañía es sólida.)

**Ya resueltos en notebooks anteriores (consolidado aquí):**

- **Sesgo por flota (91,2% Tercero):** explicado por capacidad — la
  flota propia es de solo ~127 camiones con tope diario de viajes (ver
  notebook `04`, Paso 4). No es preferencia de negocio, es restricción
  física.
- **Sesgo del segmento sin clasificar (7.496 filas sin datos de
  equipo):** estructural, no defecto — aplica solo a posiciones
  PKG/PRD por diseño del modelo de datos (ver notebook `02`, §5.1).
- **Sesgo temporal:** abril 2026 es un mes parcial (solo 7 días, ~523
  registros) — no debe compararse contra meses completos ni incluirse
  en tendencias sin marcarlo (ya aplicado desde el notebook `03`).
- **Sesgo de granularidad:** `FO` no es clave única — 128 valores
  repetidos, consistentes con carga consolidada de varios clientes en
  un mismo viaje de vehículo (ver notebook `02`, §4.2). Cualquier
  conteo de "número de viajes" debe tener esto en cuenta.

**Hallazgo lateral, sin explorar:** `FactFWOFUFO` tiene una columna
`Importe Flete SICE FO` (el valor de flete calculado directamente por
SICE-TAC) — útil en el futuro para comparar contra `Flete Producción
FO` y ver qué tan cerca sigue la empresa el piso SICE-TAC real (conecta
con el hallazgo del salto de marzo 2026 del notebook `05`).



## Síntesis del notebook 06 — Outliers, anomalías y sesgos

**Objetivo cumplido (Fases 9 y 10, §10-11 del plan):** distinguir error
de dato de caso legítimo, y entender de quién habla este dataset y de
quién no.

**Outliers segmentados (Paso 2):** segmentar por tipo de vehículo
encontró 360 outliers de flete (vs. 51 con el método global del
notebook `03`) — más sensible, no un error, porque compara cada vehículo
contra su propio rango típico. Caso de estudio (`TC_CAMNETA`, 18,4% de
outliers) se dividió en dos: outliers altos = viajes largos legítimos
(explicados por distancia); outliers bajos (todos en exactamente
$800.000) = **valor de relleno del sistema**, confirmado consultando
directamente `tblSAPTarifas` (aparece en decenas de combinaciones
ruta/peso/vehículo sin relación entre sí). Limitación anotada: el
método segmenta por una sola variable, no controla distancia — un
chequeo multivariado (`IsolationForest`) podría descartar varios de
los 360 como explicados por otra variable.

**Casos puntuales (Paso 3):** el flete de $39,6M-$46,8M en
Buenaventura→Cota es **error_dato confirmado** (verificado contra la
tarifa oficial: el techo legítimo de esa ruta es $7,4M). El flete
negativo (-$49.978) queda como `por_confirmar` (posible nota crédito).
Las 16 distancias en 0 se dividen en 3 ya conocidas (descontenerización)
y 13 con campo faltante genuino.

**Concentración de clientes (Paso 4):** top 10 = 39,7% del volumen,
pero HHI = 331 (no concentrado) — el negocio tiene una base de clientes
genuinamente amplia (537 clientes), un cliente algo más grande que el
resto (14,7%) no domina el conjunto.

**Sesgos (Pasos 5-6):**
- Cobertura geográfica: confirmado con el negocio — este set es solo
  Buenaventura, el resto de la operación nacional se evaluará con otro
  dataset después.
- Selección y supervivencia: **confirmados con evidencia directa** en
  `FactFWOFUFO` — existen 8.034 FO anulados y 115.216 en tratamiento en
  la misma ventana (compañía completa); el extracto casi con certeza
  solo contiene FOs `Concluido`/`Ejecutado`. El dataset solo representa
  viajes que llegaron a un final exitoso.
- Flota, segmento sin clasificar, temporal y granularidad: ya resueltos
  en notebooks anteriores, solo consolidados aquí.

**Qué le queda listo al cierre general (`07_sintesis_y_limitaciones.ipynb`):**
- Anexo de outliers con FO listados y su clasificación
  (error_dato/caso_legítimo/por_confirmar) — ya tenemos los principales.
- Lista de limitaciones lista para consolidar (§12 del plan): cobertura
  geográfica, selección/supervivencia, granularidad, calidad de
  `Tarifa Venta al Cliente`, mes parcial de abril 2026.
- Hallazgo lateral sin explorar: `Importe Flete SICE FO` en
  `FactFWOFUFO`, comparable contra `Flete Producción FO` si se retoma
  el análisis del ajuste de marzo 2026.
